In [1]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report
import pandas as pd
import copy
import joblib
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

CONFIG = {
    "immature_dir": r"E:\TSG\jupyterlab\machine learning image\TL SOFMaugmented_data\augmented_immature",
    "mature_dir": r"E:\TSG\jupyterlab\machine learning image\TL SOFMaugmented_data\augmented_mature",
    "num_classes": 2,
    "freeze_backbone": True,
    "freeze_epochs": 20,
    "fine_tune_epochs": 40,
    "batch_size": 32,
    "lr_frozen": 0.001,
    "lr_unfrozen": 0.0001,
    "seed": 42,
    "train_ratio": 0.6,
    "val_ratio": 0.2,
    "test_ratio": 0.2
}

def check_gpu_available():
    if not torch.cuda.is_available():
        print("Error: No available GPU device detected.")
        print("Please ensure:")
        print("1. CUDA and cuDNN are installed")
        print("2. GPU-supported PyTorch version is installed")
        print("3. Graphics driver is up to date")
        print("\nGPU is required for training, exiting...")
        sys.exit(1)
    
    print(f"✓ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"  CUDA version: {torch.version.cuda}")
    return True

check_gpu_available()

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])

class CustomDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert('RGB')
            label = self.labels[idx]
            
            if self.transform:
                image = self.transform(image)
                
            return image, label
        except Exception as e:
            print(f"Error loading image {self.image_paths[idx]}: {e}")
            return None, None

class ParallelResNet18(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        base = models.resnet18(pretrained=True)
        num_ftrs = base.fc.in_features
        base.fc = nn.Identity()
        self.branch1 = copy.deepcopy(base)
        self.branch2 = copy.deepcopy(base)
        self.fc = nn.Linear(num_ftrs * 2, num_classes)

    def forward(self, x):
        f1 = self.branch1(x)
        f2 = self.branch2(x)
        feats = torch.cat([f1, f2], dim=1)
        out = self.fc(feats)
        return out

def set_parameter_requires_grad(model, freeze_backbone):
    if freeze_backbone:
        for param in model.branch1.parameters():
            param.requires_grad = False
        for param in model.branch2.parameters():
            param.requires_grad = False
        for param in model.fc.parameters():
            param.requires_grad = True
    else:
        for param in model.parameters():
            param.requires_grad = True

def load_and_split_data():
    immature_dir = CONFIG["immature_dir"]
    mature_dir = CONFIG["mature_dir"]
    
    if not os.path.exists(immature_dir):
        print(f"Error: Path {immature_dir} does not exist")
        sys.exit(1)
    if not os.path.exists(mature_dir):
        print(f"Error: Path {mature_dir} does not exist")
        sys.exit(1)
    
    immature_paths = []
    mature_paths = []
    for img_name in os.listdir(immature_dir):
        if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            immature_paths.append(os.path.join(immature_dir, img_name))
    
    for img_name in os.listdir(mature_dir):
        if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            mature_paths.append(os.path.join(mature_dir, img_name))
    
    all_paths = immature_paths + mature_paths
    all_labels = [0] * len(immature_paths) + [1] * len(mature_paths)
    
    print(f"\nData Statistics:")
    print(f"Immature images: {len(immature_paths)}")
    print(f"Mature images: {len(mature_paths)}")
    print(f"Total images: {len(all_paths)}")
    
    train_paths, rest_paths, train_labels, rest_labels = train_test_split(
        all_paths, all_labels, 
        test_size=CONFIG["val_ratio"] + CONFIG["test_ratio"], 
        random_state=CONFIG["seed"], 
        stratify=all_labels
    )
    
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        rest_paths, rest_labels, 
        test_size=CONFIG["test_ratio"] / (CONFIG["val_ratio"] + CONFIG["test_ratio"]),
        random_state=CONFIG["seed"], 
        stratify=rest_labels
    )
    
    print(f"\nData Split Results (6:2:2):")
    print(f"Train set: {len(train_paths)} images ({len(train_paths)/len(all_paths)*100:.1f}%)")
    print(f"Val set: {len(val_paths)} images ({len(val_paths)/len(all_paths)*100:.1f}%)")
    print(f"Test set: {len(test_paths)} images ({len(test_paths)/len(all_paths)*100:.1f}%)")
    
    return train_paths, val_paths, test_paths, train_labels, val_labels, test_labels

def get_transforms():
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])
    
    val_test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])
    
    return train_transform, val_test_transform

def calculate_metrics(all_labels, all_predictions):
    accuracy = np.mean(np.array(all_labels) == np.array(all_predictions))
    precision = precision_score(all_labels, all_predictions, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_predictions, average='weighted', zero_division=0)
    f1 = f1_score(all_labels, all_predictions, average='weighted', zero_division=0)
    precision_per_class = precision_score(all_labels, all_predictions, average=None, zero_division=0)
    recall_per_class = recall_score(all_labels, all_predictions, average=None, zero_division=0)
    f1_per_class = f1_score(all_labels, all_predictions, average=None, zero_division=0)
    cm = confusion_matrix(all_labels, all_predictions)
    report = classification_report(all_labels, all_predictions, 
                                  target_names=['immature', 'mature'], 
                                  output_dict=True, zero_division=0)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'precision_per_class': precision_per_class,
        'recall_per_class': recall_per_class,
        'f1_per_class': f1_per_class,
        'confusion_matrix': cm,
        'classification_report': report
    }

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_predictions = []
    all_labels = []
    
    progress_bar = tqdm(dataloader, desc='Training')
    for images, labels in progress_bar:
        if images is None:
            continue
            
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        batch_acc = (predicted == labels).sum().item() / labels.size(0)
        progress_bar.set_postfix({'Loss': running_loss/len(dataloader), 'Acc': batch_acc})
    
    epoch_loss = running_loss / len(dataloader)
    metrics = calculate_metrics(all_labels, all_predictions)
    metrics['loss'] = epoch_loss
    
    return epoch_loss, metrics

def evaluate_model(model, dataloader, criterion, device, dataset_name="Dataset"):
    model.eval()
    running_loss = 0.0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            if images is None:
                continue
                
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(dataloader)
    metrics = calculate_metrics(all_labels, all_predictions)
    metrics['loss'] = epoch_loss
    
    print(f"\n{dataset_name} Results:")
    print(f"Loss: {epoch_loss:.4f} | Acc: {metrics['accuracy']:.4f} | F1: {metrics['f1']:.4f}")
    print(f"Confusion Matrix:\n{metrics['confusion_matrix']}")
    
    return epoch_loss, metrics

def train_transfer_learning():
    train_paths, val_paths, test_paths, train_labels, val_labels, test_labels = load_and_split_data()
    
    train_transform, val_test_transform = get_transforms()
    
    train_dataset = CustomDataset(train_paths, train_labels, train_transform)
    val_dataset = CustomDataset(val_paths, val_labels, val_test_transform)
    test_dataset = CustomDataset(test_paths, test_labels, val_test_transform)
    
    def collate_fn(batch):
        batch = [x for x in batch if x[0] is not None]
        if len(batch) == 0:
            return torch.tensor([]), torch.tensor([])
        return torch.utils.data.dataloader.default_collate(batch)
    
    train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False, collate_fn=collate_fn)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nUsing device: {device}")
    
    model = ParallelResNet18(num_classes=CONFIG["num_classes"]).to(device)
    
    print("\n" + "="*60)
    print("Stage 1: Freeze Backbone, Train Classifier Head")
    print("="*60)
    
    set_parameter_requires_grad(model, freeze_backbone=True)
    
    optimizer = optim.SGD(filter(lambda p: p.requires_grad, model.parameters()), 
                          lr=CONFIG["lr_frozen"], momentum=0.9, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
    
    best_val_acc = 0.0
    best_model_state = None
    
    for epoch in range(CONFIG["freeze_epochs"]):
        print(f"\nFrozen Training Epoch {epoch+1}/{CONFIG['freeze_epochs']}")
        train_loss, train_metrics = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_metrics = evaluate_model(model, val_loader, criterion, device, "Validation Set")
        scheduler.step()
        
        if val_metrics['accuracy'] > best_val_acc:
            best_val_acc = val_metrics['accuracy']
            best_model_state = copy.deepcopy(model.state_dict())
    
    model.load_state_dict(best_model_state)
    print(f"\nFrozen training completed, best validation accuracy: {best_val_acc:.4f}")
    
    print("\n" + "="*60)
    print("Stage 2: Unfreeze Backbone, Fine-tune Entire Model")
    print("="*60)
    
    set_parameter_requires_grad(model, freeze_backbone=False)
    
    optimizer = optim.SGD(model.parameters(), 
                          lr=CONFIG["lr_unfrozen"], momentum=0.9, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.1)
    
    for epoch in range(CONFIG["fine_tune_epochs"]):
        print(f"\nFine-tune Epoch {epoch+1}/{CONFIG['fine_tune_epochs']}")
        train_loss, train_metrics = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_metrics = evaluate_model(model, val_loader, criterion, device, "Validation Set")
        scheduler.step()
        
        if val_metrics['accuracy'] > best_val_acc:
            best_val_acc = val_metrics['accuracy']
            best_model_state = copy.deepcopy(model.state_dict())
    
    print("\n" + "="*60)
    print("Final Test (Test Set)")
    print("="*60)
    
    model.load_state_dict(best_model_state)
    test_loss, test_metrics = evaluate_model(model, test_loader, criterion, device, "Test Set")
    
    print("\n" + "="*60)
    print("Saving Model and Results")
    print("="*60)
    
    torch.save({
        'model_state_dict': best_model_state,
        'config': CONFIG,
        'test_metrics': test_metrics,
        'class_names': ['immature', 'mature']
    }, 'transfer_learning_parallel_resnet18.pth')
    
    results_df = pd.DataFrame({
        'Dataset': ['Train Set', 'Validation Set', 'Test Set'],
        'Loss': [train_loss, val_loss, test_loss],
        'Accuracy': [train_metrics['accuracy'], val_metrics['accuracy'], test_metrics['accuracy']],
        'F1 Score': [train_metrics['f1'], val_metrics['f1'], test_metrics['f1']],
        'Precision': [train_metrics['precision'], val_metrics['precision'], test_metrics['precision']],
        'Recall': [train_metrics['recall'], val_metrics['recall'], test_metrics['recall']]
    })
    results_df.to_excel('transfer_learning_results.xlsx', index=False)
    
    print("\n✓ Transfer learning training completed!")
    print(f"✓ Best validation accuracy: {best_val_acc:.4f}")
    print(f"✓ Test set accuracy: {test_metrics['accuracy']:.4f}")
    print(f"✓ Model saved to: transfer_learning_parallel_resnet18.pth")
    print(f"✓ Results saved to: transfer_learning_results.xlsx")
    
    return model, test_metrics

if __name__ == "__main__":
    model, test_metrics = train_transfer_learning()

✓ GPU available: NVIDIA GeForce RTX 5070 Ti
  VRAM: 17.09 GB
  CUDA version: 11.8

Data Statistics:
Immature images: 1780
Mature images: 1060
Total images: 2840

Data Split Results (6:2:2):
Train set: 1704 images (60.0%)
Val set: 568 images (20.0%)
Test set: 568 images (20.0%)

Using device: cuda

Stage 1: Freeze Backbone, Train Classifier Head

Frozen Training Epoch 1/20


Training: 100%|██████████████████████████████████████████████████| 54/54 [03:12<00:00,  3.57s/it, Loss=0.547, Acc=0.75]



Validation Set Results:
Loss: 0.5010 | Acc: 0.7852 | F1: 0.7879
Confusion Matrix:
[[277  79]
 [ 43 169]]

Frozen Training Epoch 2/20


Training: 100%|█████████████████████████████████████████████████| 54/54 [03:00<00:00,  3.33s/it, Loss=0.485, Acc=0.875]



Validation Set Results:
Loss: 0.5114 | Acc: 0.7271 | F1: 0.6997
Confusion Matrix:
[[331  25]
 [130  82]]

Frozen Training Epoch 3/20


...


Validation Set Results:
Loss: 0.2661 | Acc: 0.8891 | F1: 0.8873
Confusion Matrix:
[[338  18]
 [ 45 167]]

Fine-tune Epoch 39/40


Training: 100%|██████████████████████████████████████████████████| 54/54 [03:02<00:00,  3.38s/it, Loss=0.211, Acc=0.75]



Validation Set Results:
Loss: 0.2757 | Acc: 0.8908 | F1: 0.8889
Confusion Matrix:
[[340  16]
 [ 46 166]]

Fine-tune Epoch 40/40


Training: 100%|█████████████████████████████████████████████████| 54/54 [03:01<00:00,  3.36s/it, Loss=0.204, Acc=0.875]



Validation Set Results:
Loss: 0.2717 | Acc: 0.8873 | F1: 0.8853
Confusion Matrix:
[[339  17]
 [ 47 165]]

Final Test (Test Set)

Test Set Results:
Loss: 0.2342 | Acc: 0.9014 | F1: 0.9008
Confusion Matrix:
[[334  22]
 [ 34 178]]

Saving Model and Results

✓ Transfer learning training completed!
✓ Best validation accuracy: 0.8996
✓ Test set accuracy: 0.9014
✓ Model saved to: transfer_learning_parallel_resnet18.pth
✓ Results saved to: transfer_learning_results.xlsx


In [2]:
import os
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm import tqdm
import joblib
import copy
from torchvision import transforms, models

CONFIG = {
    "tl_model_path": "transfer_learning_parallel_resnet18.pth",
    "immature_dir": r"E:\TSG\jupyterlab\machine learning image\TL SOFMaugmented_data\augmented_immature",
    "mature_dir": r"E:\TSG\jupyterlab\machine learning image\TL SOFMaugmented_data\augmented_mature",
    "num_classes": 2,
    "batch_size": 32,
    "seed": 42
}

class CustomDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert('RGB')
            label = self.labels[idx]
            
            if self.transform:
                image = self.transform(image)
                
            return image, label
        except Exception as e:
            print(f"Error loading image {self.image_paths[idx]}: {e}")
            return None, None

class ParallelResNet18(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        base = models.resnet18(pretrained=True)
        num_ftrs = base.fc.in_features
        base.fc = nn.Identity()
        self.branch1 = copy.deepcopy(base)
        self.branch2 = copy.deepcopy(base)
        self.fc = nn.Linear(num_ftrs * 2, num_classes)

    def forward(self, x):
        f1 = self.branch1(x)
        f2 = self.branch2(x)
        feats = torch.cat([f1, f2], dim=1)
        out = self.fc(feats)
        return out

def create_parallel_resnet18_model(num_classes=2):
    model = ParallelResNet18(num_classes=num_classes)
    return model

def extract_features_from_model(model, dataloader, device):
    model.eval()
    all_features = []
    all_labels = []
    
    with torch.no_grad():
        progress_bar = tqdm(dataloader, desc='Extracting Features')
        for images, labels in progress_bar:
            if images is None or len(images) == 0:
                continue
            images = images.to(device)
            
            f1 = model.branch1(images)
            f2 = model.branch2(images)
            features = torch.cat([f1, f2], dim=1)
            
            all_features.extend(features.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    return np.array(all_features), np.array(all_labels)

def load_tl_data():
    from sklearn.model_selection import train_test_split
    
    immature_paths = []
    mature_paths = []
    for img_name in os.listdir(CONFIG["immature_dir"]):
        if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            immature_paths.append(os.path.join(CONFIG["immature_dir"], img_name))
    for img_name in os.listdir(CONFIG["mature_dir"]):
        if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            mature_paths.append(os.path.join(CONFIG["mature_dir"], img_name))
    
    all_paths = immature_paths + mature_paths
    all_labels = [0] * len(immature_paths) + [1] * len(mature_paths)
    
    train_paths, rest_paths, train_labels, rest_labels = train_test_split(
        all_paths, all_labels, test_size=0.4, random_state=CONFIG["seed"], stratify=all_labels
    )
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        rest_paths, rest_labels, test_size=0.5, random_state=CONFIG["seed"], stratify=rest_labels
    )
    
    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    return {
        "train_paths": train_paths, "val_paths": val_paths, "test_paths": test_paths,
        "train_labels": train_labels, "val_labels": val_labels, "test_labels": test_labels,
        "transform": test_transform
    }

def extract_and_save_features(train_mode=True):
    feat_save_path = 'transfer_learning_extracted_features_complete.pkl'
    
    if not train_mode and os.path.exists(feat_save_path):
        print("Loading saved transfer learning features...")
        try:
            features_data = joblib.load(feat_save_path)
            
            print("✓ Features loaded successfully!")
            print(f"Train features shape: {features_data['train_features'].shape}")
            print(f"Val features shape: {features_data['val_features'].shape}")
            print(f"Test features shape: {features_data['test_features'].shape}")
            
            return features_data
            
        except Exception as e:
            print(f"Error loading features: {e}")
            print("Re-extracting features...")
            train_mode = True
    
    if not os.path.exists(CONFIG["tl_model_path"]):
        print(f"Error: Transfer learning model file not found {CONFIG['tl_model_path']}")
        print("Please run the first part of transfer learning to train the model first")
        return None
    
    print("Loading transfer learning Parallel ResNet18 model...")
    checkpoint = torch.load(CONFIG["tl_model_path"], 
                          map_location='cuda' if torch.cuda.is_available() else 'cpu')
    
    model = create_parallel_resnet18_model(num_classes=CONFIG["num_classes"])
    model.load_state_dict(checkpoint['model_state_dict'])
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    print("✓ Transfer learning model loaded successfully")
    
    print("\nLoading transfer learning dataset...")
    tl_data = load_tl_data()
    test_transform = tl_data["transform"]
    print("✓ Dataset loaded successfully")
    
    def collate_fn(batch):
        batch = [x for x in batch if x[0] is not None]
        if len(batch) == 0:
            return torch.tensor([]), torch.tensor([])
        return torch.utils.data.dataloader.default_collate(batch)
    
    print("\nCreating datasets for feature extraction...")
    train_dataset_feat = CustomDataset(tl_data["train_paths"], tl_data["train_labels"], test_transform)
    val_dataset_feat = CustomDataset(tl_data["val_paths"], tl_data["val_labels"], test_transform)
    test_dataset_feat = CustomDataset(tl_data["test_paths"], tl_data["test_labels"], test_transform)
    
    train_loader_feat = DataLoader(train_dataset_feat, batch_size=CONFIG["batch_size"], shuffle=False, collate_fn=collate_fn)
    val_loader_feat = DataLoader(val_dataset_feat, batch_size=CONFIG["batch_size"], shuffle=False, collate_fn=collate_fn)
    test_loader_feat = DataLoader(test_dataset_feat, batch_size=CONFIG["batch_size"], shuffle=False, collate_fn=collate_fn)
    
    print("\nExtracting training set features...")
    train_features, train_feat_labels = extract_features_from_model(model, train_loader_feat, device)
    
    print("Extracting validation set features...")
    val_features, val_feat_labels = extract_features_from_model(model, val_loader_feat, device)
    
    print("Extracting test set features...")
    test_features, test_feat_labels = extract_features_from_model(model, test_loader_feat, device)
    
    print(f"\nFeature extraction completed:")
    print(f"Train features shape: {train_features.shape}")
    print(f"Val features shape: {val_features.shape}")
    print(f"Test features shape: {test_features.shape}")
    
    print("\n" + "="*60)
    print("Saving transfer learning features...")
    print("="*60)
    
    features_data = {
        'train_features': train_features,
        'val_features': val_features,
        'test_features': test_features,
        'train_labels': train_feat_labels,
        'val_labels': val_feat_labels,
        'test_labels': test_feat_labels,
        'train_paths': tl_data["train_paths"],
        'val_paths': tl_data["val_paths"],
        'test_paths': tl_data["test_paths"],
        'feature_dim': train_features.shape[1],
        'device': str(device)
    }
    
    joblib.dump(features_data, feat_save_path)
    print(f"✓ Feature data saved to: {feat_save_path}")
    
    np.save('tl_train_features.npy', train_features)
    np.save('tl_val_features.npy', val_features)
    np.save('tl_test_features.npy', test_features)
    np.save('tl_train_labels.npy', train_feat_labels)
    np.save('tl_val_labels.npy', val_feat_labels)
    np.save('tl_test_labels.npy', test_feat_labels)
    
    print("✓ Transfer learning feature arrays saved as .npy files")
    
    path_info = {
        'train_paths': tl_data["train_paths"],
        'val_paths': tl_data["val_paths"],
        'test_paths': tl_data["test_paths"]
    }
    joblib.dump(path_info, 'tl_image_paths_info.pkl')
    print("✓ Transfer learning image path info saved to: tl_image_paths_info.pkl")
    
    print("\n" + "="*60)
    print("Transfer learning feature extraction completed!")
    print("="*60)
    
    return features_data

if __name__ == "__main__":
    feat_save_path = 'transfer_learning_extracted_features_complete.pkl'
    if os.path.exists(feat_save_path):
        print("Saved transfer learning features detected. Load them? (y/n)")
        choice = input().strip().lower()
        if choice == 'y':
            features_data = extract_and_save_features(train_mode=False)
        else:
            features_data = extract_and_save_features(train_mode=True)
    else:
        features_data = extract_and_save_features(train_mode=True)

Loading transfer learning Parallel ResNet18 model...
✓ Transfer learning model loaded successfully

Loading transfer learning dataset...
✓ Dataset loaded successfully

Creating datasets for feature extraction...

Extracting training set features...


Extracting Features: 100%|█████████████████████████████████████████████████████████████| 54/54 [02:55<00:00,  3.25s/it]


Extracting validation set features...


Extracting Features: 100%|█████████████████████████████████████████████████████████████| 18/18 [00:59<00:00,  3.31s/it]


Extracting test set features...


Extracting Features: 100%|█████████████████████████████████████████████████████████████| 18/18 [00:58<00:00,  3.27s/it]


Feature extraction completed:
Train features shape: (1704, 1024)
Val features shape: (568, 1024)
Test features shape: (568, 1024)

Saving transfer learning features...
✓ Feature data saved to: transfer_learning_extracted_features_complete.pkl
✓ Transfer learning feature arrays saved as .npy files
✓ Transfer learning image path info saved to: tl_image_paths_info.pkl

Transfer learning feature extraction completed!


In [3]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
import warnings
import pickle
warnings.filterwarnings('ignore')

CONFIG = {
    "tl_feat_path": "transfer_learning_extracted_features_complete.pkl",
    "result_prefix": "tl_",
    "sample_ratio": 0.5,
    "train_test_split_ratio": 0.8,
    "random_seed": 42
}

try:
    from minisom import MiniSom
    MINISOM_AVAILABLE = True
except ImportError:
    print("MiniSom library not installed, installing...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "minisom"])
    from minisom import MiniSom
    MINISOM_AVAILABLE = True

class TargetedSOFMClassifier:
    def __init__(self, grid_size=(12, 12), input_len=1024, sigma=1.2, learning_rate=0.3):
        self.grid_size = grid_size
        self.input_len = input_len
        self.sigma = sigma
        self.learning_rate = learning_rate
        self.som = None
        self.weights = None
        
    def fit(self, X, n_iterations=20000, random_seed=42):
        self.som = MiniSom(
            x=self.grid_size[0],
            y=self.grid_size[1],
            input_len=self.input_len,
            sigma=self.sigma,
            learning_rate=self.learning_rate,
            random_seed=random_seed
        )
        
        self.som.random_weights_init(X)
        self.som.train_random(X, n_iterations)
        
        self.weights = self.som.get_weights().reshape(-1, self.input_len)
        return self
    
    def predict_with_maturity_focus(self, X, labels, n_immature_clusters=4, n_mature_clusters=2, 
                                    min_mature_purity=0.7, random_seed=42):
        if self.som is None:
            raise ValueError("SOFM not trained")
        
        weights = self.weights
        
        total_clusters = 6
        kmeans = KMeans(n_clusters=total_clusters, random_state=random_seed, n_init=20)
        clustering_labels = kmeans.fit_predict(weights)
        
        predictions = []
        for sample in X:
            winner = self.som.winner(sample)
            winner_idx = winner[0] * self.grid_size[1] + winner[1]
            cluster_label = clustering_labels[winner_idx]
            predictions.append(cluster_label)
        
        predictions = np.array(predictions)
        
        mature_clusters_found = 0
        purity_sum = 0
        
        for cluster_id in np.unique(predictions):
            cluster_mask = predictions == cluster_id
            if np.sum(cluster_mask) > 0:
                mature_count = np.sum((labels == 1) & cluster_mask)
                total_count = np.sum(cluster_mask)
                purity = mature_count / total_count
                
                if purity >= min_mature_purity:
                    mature_clusters_found += 1
                    purity_sum += purity
        
        print(f"Using 6-cluster configuration: Found {mature_clusters_found} mature clusters (target: 2)")
        return predictions, kmeans, clustering_labels

def post_process_predictions_for_maturity(predictions, features, true_labels, 
                                          target_mature_clusters=2, min_purity=0.7):
    print("\nPost-processing clustering results to improve mature cluster purity...")
    
    cluster_purities = {}
    for cluster_id in np.unique(predictions):
        cluster_mask = predictions == cluster_id
        if np.sum(cluster_mask) > 0:
            mature_count = np.sum((true_labels == 1) & cluster_mask)
            total_count = np.sum(cluster_mask)
            purity = mature_count / total_count
            cluster_purities[cluster_id] = {
                'purity': purity,
                'mature_count': mature_count,
                'total_count': total_count,
                'features': features[cluster_mask],
                'indices': np.where(cluster_mask)[0]
            }
    
    mature_candidates = []
    for cluster_id, stats in cluster_purities.items():
        if stats['purity'] >= min_purity:
            mature_candidates.append((cluster_id, stats['purity'], stats['mature_count'], stats['total_count']))
    
    mature_candidates.sort(key=lambda x: x[1], reverse=True)
    
    print(f"Current high-purity mature clusters: {len(mature_candidates)}")
    for cluster_id, purity, mature_count, total_count in mature_candidates:
        print(f"  Cluster {cluster_id}: Purity {purity:.1%}, Mature {mature_count}/{total_count}")
    
    if len(mature_candidates) < target_mature_clusters:
        print(f"Insufficient mature clusters ({len(mature_candidates)} < {target_mature_clusters}), optimizing...")
        
        near_mature_candidates = []
        for cluster_id, stats in cluster_purities.items():
            if 0.5 <= stats['purity'] < min_purity:
                near_mature_candidates.append((cluster_id, stats['purity'], stats))
        
        near_mature_candidates.sort(key=lambda x: x[1], reverse=True)
        
        for cluster_id, purity, stats in near_mature_candidates[:target_mature_clusters - len(mature_candidates)]:
            print(f"  Optimizing cluster {cluster_id} (Current purity: {purity:.1%})...")
            
            if len(stats['indices']) > 10:
                sub_kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
                sub_features = stats['features']
                sub_labels = sub_kmeans.fit_predict(sub_features)
                
                for sub_cluster in [0, 1]:
                    sub_mask = sub_labels == sub_cluster
                    if np.sum(sub_mask) > 0:
                        sub_mature_count = np.sum((true_labels[stats['indices']] == 1) & sub_mask)
                        sub_total = np.sum(sub_mask)
                        sub_purity = sub_mature_count / sub_total if sub_total > 0 else 0
                        
                        if sub_purity >= min_purity and sub_total >= 20:
                            print(f"    Sub-cluster {sub_cluster}: Purity {sub_purity:.1%}, Samples {sub_total}")
                            
                            for i, idx in enumerate(stats['indices']):
                                if sub_labels[i] == sub_cluster:
                                    pass
                                else:
                                    predictions[idx] = -1
    
    return predictions

def create_maturity_levels_with_purity_guarantee(predictions, true_labels, 
                                                 target_immature=4, target_mature=2, 
                                                 min_immature_purity=0.7, min_mature_purity=0.6):
    print(f"\nCreating maturity levels (Forced target: {target_immature} Immature + {target_mature} Mature)...")
    
    cluster_stats = {}
    for cluster_id in np.unique(predictions):
        if cluster_id == -1:
            continue
            
        cluster_mask = predictions == cluster_id
        if np.sum(cluster_mask) > 0:
            immature_count = np.sum((true_labels == 0) & cluster_mask)
            mature_count = np.sum((true_labels == 1) & cluster_mask)
            total_count = np.sum(cluster_mask)
            
            immature_ratio = immature_count / total_count
            mature_ratio = mature_count / total_count
            
            if immature_ratio >= mature_ratio:
                dominant_class = 0
                purity = immature_ratio
            else:
                dominant_class = 1
                purity = mature_ratio
            
            cluster_stats[cluster_id] = {
                'count': total_count,
                'immature_count': immature_count,
                'mature_count': mature_count,
                'immature_ratio': immature_ratio,
                'mature_ratio': mature_ratio,
                'purity': purity,
                'dominant_class': dominant_class
            }
    
    immature_clusters = []
    mature_clusters = []
    
    for cluster_id, stats in cluster_stats.items():
        if stats['dominant_class'] == 0:
            immature_clusters.append((cluster_id, stats['purity'], stats['count']))
        else:
            mature_clusters.append((cluster_id, stats['purity'], stats['count']))
    
    immature_clusters.sort(key=lambda x: x[1], reverse=True)
    mature_clusters.sort(key=lambda x: x[1], reverse=True)
    
    print(f"Available clusters: {len(immature_clusters)} Immature, {len(mature_clusters)} Mature")
    
    selected_immature = immature_clusters[:target_immature]
    if len(selected_immature) < target_immature:
        additional_immature = [c for c in immature_clusters[target_immature:] if c[1] >= 0.5]
        selected_immature += additional_immature[:target_immature - len(selected_immature)]
    
    selected_mature = mature_clusters[:target_mature]
    if len(selected_mature) < target_mature:
        additional_mature = [c for c in mature_clusters[target_mature:] if c[1] >= 0.5]
        selected_mature += additional_mature[:target_mature - len(selected_mature)]
    
    if len(selected_immature) < target_immature:
        print(f"Warning: Insufficient immature clusters, supplementing from low-purity clusters")
        cross_clusters = [c for c in mature_clusters if c[1] < 0.5]
        selected_immature += cross_clusters[:target_immature - len(selected_immature)]
    
    if len(selected_mature) < target_mature:
        print(f"Warning: Insufficient mature clusters, supplementing from low-purity clusters")
        cross_clusters = [c for c in immature_clusters if c[1] < 0.5]
        selected_mature += cross_clusters[:target_mature - len(selected_mature)]
    
    cluster_to_level = {}
    level_descriptions = {}
    
    for i, (cluster_id, purity, count) in enumerate(selected_immature[:4]):
        level = -(i + 1)
        cluster_to_level[cluster_id] = level
        purity_status = "High Purity" if purity >= min_immature_purity else "Medium Purity" if purity >=0.6 else "Low Purity"
        level_descriptions[level] = f"Immature Level {abs(level)} {purity_status} (Purity: {purity:.1%}, Samples: {count})"
    
    for i, (cluster_id, purity, count) in enumerate(selected_mature[:2]):
        level = i + 1
        cluster_to_level[cluster_id] = level
        purity_status = "High Purity" if purity >= min_mature_purity else "Medium Purity" if purity >=0.6 else "Low Purity"
        level_descriptions[level] = f"Mature Level {level} {purity_status} (Purity: {purity:.1%}, Samples: {count})"
    
    maturity_levels = np.array([cluster_to_level.get(label, 0) for label in predictions])
    
    unique_levels = np.unique(maturity_levels)
    immature_levels = [l for l in unique_levels if l < 0]
    mature_levels = [l for l in unique_levels if l > 0]
    
    print(f"\nFinal Level Assignment (Forced 4 Immature + 2 Mature):")
    print(f"  Immature Levels: {len(immature_levels)}")
    for level in sorted(immature_levels):
        print(f"    Level {level}: {level_descriptions.get(level, 'Supplemented Low-Purity Level')}")
    
    print(f"  Mature Levels: {len(mature_levels)}")
    for level in sorted(mature_levels):
        print(f"    Level {level}: {level_descriptions.get(level, 'Supplemented Low-Purity Level')}")
    
    return cluster_to_level, level_descriptions, maturity_levels

def main_direct_maturity_optimization():
    print("SOFM Direct Maturity Optimization System (Transfer Learning Version)")
    print("="*60)
    print("Objective: 50% overall sampling (Train+Test) + 4 Immature + 2 Mature Clusters + No Validation Set")
    print("="*60)
    
    print("\n1. Loading transfer learning feature data...")
    if not os.path.exists(CONFIG["tl_feat_path"]):
        print(f"Error: Transfer learning feature file not found {CONFIG['tl_feat_path']}")
        print("Please run the second part of transfer learning to extract features first")
        return None
    
    features_data = joblib.load(CONFIG["tl_feat_path"])
    
    train_features = features_data['train_features']
    test_features = features_data['test_features']
    train_labels = features_data['train_labels']
    test_labels = features_data['test_labels']
    train_paths = features_data['train_paths']
    test_paths = features_data['test_paths']
    
    print(f"Original Train Features: {train_features.shape}")
    print(f"Original Test Features: {test_features.shape}")
    total_original_samples = len(train_labels) + len(test_labels)
    print(f"Original Total Samples (Train+Test): {total_original_samples}")
    
    print(f"\n2. Stratified sampling {CONFIG['sample_ratio']*100}% of overall Train+Test data...")
    
    all_features = np.concatenate([train_features, test_features], axis=0)
    all_labels = np.concatenate([train_labels, test_labels], axis=0)
    all_paths = np.concatenate([train_paths, test_paths], axis=0)
    
    print(f"Total samples after merging: {len(all_labels)}")
    
    sampled_features, _, sampled_labels, _, sampled_paths, _ = train_test_split(
        all_features, 
        all_labels, 
        all_paths,
        test_size=1 - CONFIG["sample_ratio"],
        random_state=CONFIG["random_seed"],
        stratify=all_labels
    )
    
    new_train_features, new_test_features, new_train_labels, new_test_labels, new_train_paths, new_test_paths = train_test_split(
        sampled_features,
        sampled_labels,
        sampled_paths,
        test_size=1 - CONFIG["train_test_split_ratio"],
        random_state=CONFIG["random_seed"],
        stratify=sampled_labels
    )
    
    train_features = new_train_features
    train_labels = new_train_labels
    train_paths = new_train_paths
    test_features = new_test_features
    test_labels = new_test_labels
    test_paths = new_test_paths
    
    print(f"Total samples after 50% overall sampling: {len(sampled_labels)} (Target: {total_original_samples * CONFIG['sample_ratio']:.0f})")
    print(f"Sampled Train Features: {train_features.shape}")
    print(f"Sampled Test Features: {test_features.shape}")
    
    original_immature_ratio = np.sum(all_labels == 0) / len(all_labels)
    sampled_immature_ratio = np.sum(sampled_labels == 0) / len(sampled_labels)
    print(f"Immature Ratio - Original Merged: {original_immature_ratio:.1%} | Sampled: {sampled_immature_ratio:.1%}")
    
    print("\n3. Training SOFM model...")
    
    best_params = {
        'grid_size': (12, 12),
        'sigma': 1.0,
        'learning_rate': 0.25,
        'n_iterations': 20000,
        'n_immature_clusters': 4,
        'n_mature_clusters': 2,
        'min_mature_purity': 0.6
    }
    
    sofm = TargetedSOFMClassifier(
        grid_size=best_params['grid_size'],
        input_len=train_features.shape[1],
        sigma=best_params['sigma'],
        learning_rate=best_params['learning_rate']
    )
    
    sofm.fit(train_features, n_iterations=best_params['n_iterations'])
    
    print(f"\n4. Performing maturity-focused clustering (Forced 6 clusters)...")
    train_predictions, kmeans_model, neuron_labels = sofm.predict_with_maturity_focus(
        train_features, train_labels,
        n_immature_clusters=best_params['n_immature_clusters'],
        n_mature_clusters=best_params['n_mature_clusters'],
        min_mature_purity=best_params['min_mature_purity']
    )
    
    train_predictions = post_process_predictions_for_maturity(
        train_predictions, train_features, train_labels,
        target_mature_clusters=best_params['n_mature_clusters'],
        min_purity=best_params['min_mature_purity']
    )
    
    print("\n5. Predicting on test set...")
    test_predictions = []
    for sample in test_features:
        winner = sofm.som.winner(sample)
        winner_idx = winner[0] * sofm.grid_size[1] + winner[1]
        cluster_label = neuron_labels[winner_idx]
        test_predictions.append(cluster_label)
    test_predictions = np.array(test_predictions)
    
    print("\n6. Creating maturity levels (Forced 4 Immature + 2 Mature)...")
    cluster_to_level, level_descriptions, train_maturity_levels = create_maturity_levels_with_purity_guarantee(
        train_predictions, train_labels,
        target_immature=best_params['n_immature_clusters'],
        target_mature=best_params['n_mature_clusters'],
        min_immature_purity=0.7,
        min_mature_purity=0.6
    )
    
    test_maturity_levels = np.array([cluster_to_level.get(label, 0) for label in test_predictions])
    
    train_classified = np.sum(train_maturity_levels != 0)
    test_classified = np.sum(test_maturity_levels != 0)
    train_coverage = train_classified / len(train_maturity_levels)
    test_coverage = test_classified / len(test_maturity_levels)
    
    print(f"\nClassification Coverage:")
    print(f"  Train Set (Overall Sampled): {train_classified}/{len(train_maturity_levels)} = {train_coverage:.1%}")
    print(f"  Test Set (Overall Sampled): {test_classified}/{len(test_maturity_levels)} = {test_coverage:.1%}")
    
    print("\n7. Generating detailed statistics...")
    
    def get_level_statistics(maturity_levels, original_labels, level_descriptions, dataset_name):
        unique_levels = np.unique(maturity_levels)
        
        immature_levels = [l for l in unique_levels if l < 0]
        mature_levels = [l for l in unique_levels if l > 0]
        
        print(f"\n{dataset_name} Level Statistics:")
        print("Level\tSamples\tOrig.Immature\tOrig.Mature\tImmature Ratio\tMature Ratio\tPurity\tDescription")
        print("-"*100)
        
        level_stats = []
        
        for level in sorted(immature_levels, reverse=True):
            level_mask = maturity_levels == level
            level_samples = np.sum(level_mask)
            
            immature_count = np.sum((original_labels == 0) & level_mask)
            mature_count = np.sum((original_labels == 1) & level_mask)
            
            immature_ratio = immature_count / level_samples if level_samples > 0 else 0
            mature_ratio = mature_count / level_samples if level_samples > 0 else 0
            purity = immature_ratio
            
            purity_symbol = "✓" if purity >= 0.8 else "⚠" if purity >= 0.7 else "✗"
            
            print(f"{purity_symbol} Level {level}\t{level_samples}\t{immature_count}\t{mature_count}\t"
                  f"{immature_ratio:>7.1%}\t{mature_ratio:>7.1%}\t{purity:>6.1%}\t{level_descriptions.get(level, '')}")
            
            level_stats.append({
                'Level': level,
                'Samples': level_samples,
                'Orig_Immature': immature_count,
                'Orig_Mature': mature_count,
                'Immature_Ratio': immature_ratio,
                'Mature_Ratio': mature_ratio,
                'Purity': purity
            })
        
        for level in sorted(mature_levels):
            level_mask = maturity_levels == level
            level_samples = np.sum(level_mask)
            
            immature_count = np.sum((original_labels == 0) & level_mask)
            mature_count = np.sum((original_labels == 1) & level_mask)
            
            immature_ratio = immature_count / level_samples if level_samples > 0 else 0
            mature_ratio = mature_count / level_samples if level_samples > 0 else 0
            purity = mature_ratio
            
            purity_symbol = "✓" if purity >= 0.7 else "⚠" if purity >= 0.6 else "✗"
            
            print(f"{purity_symbol} Level {level}\t{level_samples}\t{immature_count}\t{mature_count}\t"
                  f"{immature_ratio:>7.1%}\t{mature_ratio:>7.1%}\t{purity:>6.1%}\t{level_descriptions.get(level, '')}")
            
            level_stats.append({
                'Level': level,
                'Samples': level_samples,
                'Orig_Immature': immature_count,
                'Orig_Mature': mature_count,
                'Immature_Ratio': immature_ratio,
                'Mature_Ratio': mature_ratio,
                'Purity': purity
            })
        
        unclassified = np.sum(maturity_levels == 0)
        if unclassified > 0:
            print(f"\nUnclassified Samples: {unclassified}")
        
        return pd.DataFrame(level_stats)
    
    train_stats_df = get_level_statistics(train_maturity_levels, train_labels, level_descriptions, "Train Set (Overall Sampled)")
    test_stats_df = get_level_statistics(test_maturity_levels, test_labels, level_descriptions, "Test Set (Overall Sampled)")
    
    print("\n8. Saving results...")
    
    detailed_results = []
    
    for i in range(len(train_paths)):
        result = {
            'Dataset': 'Train Set (Overall Sampled)',
            'Image_Name': os.path.basename(train_paths[i]),
            'Original_Class': 'Immature' if train_labels[i] == 0 else 'Mature',
            'Cluster_ID': int(train_predictions[i]) if i < len(train_predictions) else -1,
            'Maturity_Level': int(train_maturity_levels[i]) if train_maturity_levels[i] != 0 else "Unclassified",
            'Level_Description': level_descriptions.get(train_maturity_levels[i], 'Unclassified')
        }
        detailed_results.append(result)
    
    for i in range(len(test_paths)):
        result = {
            'Dataset': 'Test Set (Overall Sampled)',
            'Image_Name': os.path.basename(test_paths[i]),
            'Original_Class': 'Immature' if test_labels[i] == 0 else 'Mature',
            'Cluster_ID': int(test_predictions[i]) if i < len(test_predictions) else -1,
            'Maturity_Level': int(test_maturity_levels[i]) if test_maturity_levels[i] != 0 else "Unclassified",
            'Level_Description': level_descriptions.get(test_maturity_levels[i], 'Unclassified')
        }
        detailed_results.append(result)
    
    detailed_df = pd.DataFrame(detailed_results)
    
    output_file = f"{CONFIG['result_prefix']}SOFM_50pct_Overall_Sampling_6Clusters_No_Validation.xlsx"
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        train_stats_df.to_excel(writer, sheet_name='Train_Stats', index=False)
        test_stats_df.to_excel(writer, sheet_name='Test_Stats', index=False)
        detailed_df.to_excel(writer, sheet_name='Detailed_Results', index=False)
        
        params_df = pd.DataFrame([{'Parameter': k, 'Value': v} for k, v in best_params.items()])
        params_df.to_excel(writer, sheet_name='Parameters', index=False)
        
        desc_df = pd.DataFrame([{'Level': k, 'Description': v} for k, v in level_descriptions.items()])
        desc_df.to_excel(writer, sheet_name='Level_Descriptions', index=False)
        
        sample_stats = pd.DataFrame([
            {
                'Metric': 'Original Total Samples (Train+Test)',
                'Value': total_original_samples
            },
            {
                'Metric': 'Total Samples After 50% Sampling',
                'Value': len(sampled_labels)
            },
            {
                'Metric': 'Sampled Train Samples',
                'Value': len(train_labels)
            },
            {
                'Metric': 'Sampled Test Samples',
                'Value': len(test_labels)
            },
            {
                'Metric': 'Original Immature Ratio',
                'Value': f"{original_immature_ratio:.1%}"
            },
            {
                'Metric': 'Sampled Immature Ratio',
                'Value': f"{sampled_immature_ratio:.1%}"
            }
        ])
        sample_stats.to_excel(writer, sheet_name='Sampling_Stats', index=False)
    
    print(f"\n✓ Results saved to: {output_file}")
    
    print("\n9. Saving model...")
    sofm_data = {
        'best_params': best_params,
        'train_predictions': train_predictions,
        'test_predictions': test_predictions,
        'train_maturity_levels': train_maturity_levels,
        'test_maturity_levels': test_maturity_levels,
        'cluster_to_level': cluster_to_level,
        'level_descriptions': level_descriptions,
        'train_coverage': train_coverage,
        'test_coverage': test_coverage,
        'model': sofm,
        'sample_ratio': CONFIG['sample_ratio'],
        'train_test_split_ratio': CONFIG['train_test_split_ratio'],
        'sample_seed': CONFIG['random_seed'],
        'original_total_samples': total_original_samples,
        'sampled_total_samples': len(sampled_labels)
    }
    
    with open(f"{CONFIG['result_prefix']}sofm_50pct_sampling_6clusters_no_val_optimized.pkl", 'wb') as f:
        pickle.dump(sofm_data, f)
    
    print(f"✓ Model saved to: {CONFIG['result_prefix']}sofm_50pct_sampling_6clusters_no_val_optimized.pkl")
    
    print("\n" + "="*60)
    print("Optimization Summary:")
    print("="*60)
    
    mature_purities = []
    for _, row in train_stats_df.iterrows():
        if row['Level'] > 0:
            mature_purities.append(row['Purity'])
    
    if mature_purities:
        avg_mature_purity = np.mean(mature_purities)
        print(f"Average Mature Cluster Purity: {avg_mature_purity:.1%}")
        
        if len(mature_purities) >= 2:
            print(f"Second Mature Level Purity: {mature_purities[1]:.1%}")
    
    print(f"Train Set Coverage (Overall Sampled): {train_coverage:.1%}")
    print(f"Test Set Coverage (Overall Sampled): {test_coverage:.1%}")
    print(f"Overall Sampling Rate: {CONFIG['sample_ratio']*100}% (Original {total_original_samples} → Sampled {len(sampled_labels)})")
    
    return sofm_data

if __name__ == "__main__":
    print("SOFM Direct Maturity Optimization System (Transfer Learning Version)")
    print("="*60)
    print("Config: 50% Overall Sampling (Train+Test) + 4 Immature + 2 Mature Clusters + No Validation Set")
    print("="*60)
    
    sofm_data = main_direct_maturity_optimization()

SOFM Direct Maturity Optimization System (Transfer Learning Version)
Config: 50% Overall Sampling (Train+Test) + 4 Immature + 2 Mature Clusters + No Validation Set
SOFM Direct Maturity Optimization System (Transfer Learning Version)
Objective: 50% overall sampling (Train+Test) + 4 Immature + 2 Mature Clusters + No Validation Set

1. Loading transfer learning feature data...
Original Train Features: (1704, 1024)
Original Test Features: (568, 1024)
Original Total Samples (Train+Test): 2272

2. Stratified sampling 50.0% of overall Train+Test data...
Total samples after merging: 2272
Total samples after 50% overall sampling: 1136 (Target: 1136)
Sampled Train Features: (908, 1024)
Sampled Test Features: (228, 1024)
Immature Ratio - Original Merged: 62.7% | Sampled: 62.7%

3. Training SOFM model...

4. Performing maturity-focused clustering (Forced 6 clusters)...
Using 6-cluster configuration: Found 2 mature clusters (target: 2)

Post-processing clustering results to improve mature cluster p

In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
import pickle
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import xgboost as xgb
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                           confusion_matrix, classification_report, roc_auc_score)
import warnings
warnings.filterwarnings('ignore')

CONFIG = {
    "tl_sofm_path": "tl_sofm_direct_optimized.pkl",
    "tl_feat_path": "transfer_learning_extracted_features_complete.pkl",
    "result_prefix": "tl_",
    "use_val_set": False
}

try:
    plt.rcParams['font.sans-serif'] = ['SimHei']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

class SupervisedMaturityClassifier:
    def __init__(self, random_state=42, n_jobs=-1):
        self.random_state = random_state
        self.n_jobs = n_jobs
        self.models = {}
        self.scaler = StandardScaler()
        self.best_params = {}
        self.model_performance = {}
        self.training_times = {}
        
    def load_sofm_data(self, sofm_data_path=None):
        if sofm_data_path is None:
            sofm_data_path = CONFIG["tl_sofm_path"]
        
        print("Loading transfer learning SOFM clustering results...")
        with open(sofm_data_path, 'rb') as f:
            sofm_data = pickle.load(f)
        
        features_data = joblib.load(CONFIG["tl_feat_path"])
        
        train_features = features_data['train_features']
        test_features = features_data['test_features']
        
        train_maturity_levels = sofm_data['train_maturity_levels']
        test_maturity_levels = sofm_data['test_maturity_levels']
        
        train_mask = train_maturity_levels != 0
        test_mask = test_maturity_levels != 0
        
        print(f"Labeled training samples: {np.sum(train_mask)}/{len(train_maturity_levels)}")
        print(f"Labeled test samples: {np.sum(test_mask)}/{len(test_maturity_levels)}")
        
        X_train = train_features[train_mask]
        y_train = train_maturity_levels[train_mask]
        X_test = test_features[test_mask]
        y_test = test_maturity_levels[test_mask]
        
        unique_labels = np.unique(np.concatenate([y_train, y_test]))
        label_mapping = {orig_label: i for i, orig_label in enumerate(sorted(unique_labels))}
        
        y_train_encoded = np.array([label_mapping[label] for label in y_train])
        y_test_encoded = np.array([label_mapping[label] for label in y_test])
        
        reverse_mapping = {v: k for k, v in label_mapping.items()}
        
        print(f"Original labels: {sorted(unique_labels)} -> Encoded labels: {sorted(np.unique(y_train_encoded))}")
        
        print("\nClass sample statistics:")
        for orig_label, encoded_label in label_mapping.items():
            train_count = np.sum(y_train == orig_label)
            test_count = np.sum(y_test == orig_label)
            print(f"Level {orig_label}(Encoded {encoded_label}): Train={train_count}, Test={test_count}")
        
        return X_train, X_test, y_train_encoded, y_test_encoded, label_mapping, reverse_mapping, sofm_data
    
    def prepare_data(self, X_train, X_test, y_train, y_test):
        print("\nPreparing data (Transfer Learning Version, No Validation Set)...")
        
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        print(f"Training set: {X_train_scaled.shape}, Test set: {X_test_scaled.shape}")
        
        unique_classes, class_counts = np.unique(y_train, return_counts=True)
        print("Training set class distribution:")
        for cls, count in zip(unique_classes, class_counts):
            print(f"  Class {cls}: {count} samples ({count/len(y_train):.1%})")
        
        return X_train_scaled, X_test_scaled, y_train, y_test
    
    def train_knn_simplified(self, X_train, y_train, cv_folds=3):
        print("\nTraining KNN Classifier (Simplified Search)...")
        
        import time
        start_time = time.time()
        
        param_grid = {
            'n_neighbors': [3, 5, 7],
            'weights': ['uniform', 'distance'],
            'metric': ['euclidean', 'manhattan']
        }
        
        knn = KNeighborsClassifier()
        
        random_search = RandomizedSearchCV(
            knn, param_grid, n_iter=6,
            cv=cv_folds, scoring='f1_weighted',
            n_jobs=self.n_jobs, verbose=1, random_state=self.random_state
        )
        
        random_search.fit(X_train, y_train)
        
        best_knn = random_search.best_estimator_
        self.models['knn'] = best_knn
        self.best_params['knn'] = random_search.best_params_
        
        training_time = time.time() - start_time
        self.training_times['knn'] = training_time
        
        print(f"KNN Best Params: {random_search.best_params_}")
        print(f"KNN Best Score: {random_search.best_score_:.4f}")
        print(f"KNN Training Time: {training_time:.1f}s")
        
        return best_knn
    
    def train_random_forest_simplified(self, X_train, y_train, cv_folds=3):
        print("\nTraining Random Forest Classifier (Simplified Search)...")
        
        import time
        start_time = time.time()
        
        param_grid = {
            'n_estimators': [100, 200],
            'max_depth': [10, 20, None],
            'min_samples_split': [2, 5],
            'min_samples_leaf': [1, 2],
            'class_weight': ['balanced']
        }
        
        rf = RandomForestClassifier(random_state=self.random_state)
        
        random_search = RandomizedSearchCV(
            rf, param_grid, n_iter=8,
            cv=cv_folds, scoring='f1_weighted',
            n_jobs=self.n_jobs, verbose=1, random_state=self.random_state
        )
        
        random_search.fit(X_train, y_train)
        
        best_rf = random_search.best_estimator_
        self.models['rf'] = best_rf
        self.best_params['rf'] = random_search.best_params_
        
        training_time = time.time() - start_time
        self.training_times['rf'] = training_time
        
        print(f"Random Forest Best Params: {random_search.best_params_}")
        print(f"Random Forest Best Score: {random_search.best_score_:.4f}")
        print(f"Random Forest Training Time: {training_time:.1f}s")
        
        feature_importance = best_rf.feature_importances_
        top_features = np.argsort(feature_importance)[-5:]
        print(f"Top 5 important feature indices: {top_features}")
        
        return best_rf
    
    def train_xgboost_simplified(self, X_train, y_train, cv_folds=3):
        print("\nTraining XGBoost Classifier (Simplified Search)...")
        
        import time
        start_time = time.time()
        
        param_grid = {
            'n_estimators': [100, 150],
            'max_depth': [3, 5],
            'learning_rate': [0.05, 0.1],
            'subsample': [0.8, 1.0],
            'colsample_bytree': [0.8, 1.0]
        }
        
        xgb_model = xgb.XGBClassifier(
            random_state=self.random_state,
            use_label_encoder=False,
            eval_metric='mlogloss',
            n_jobs=self.n_jobs // 2
        )
        
        random_search = RandomizedSearchCV(
            xgb_model, param_grid, n_iter=5,
            cv=cv_folds, scoring='f1_weighted',
            n_jobs=self.n_jobs, verbose=1, random_state=self.random_state
        )
        
        random_search.fit(X_train, y_train)
        
        best_xgb = random_search.best_estimator_
        self.models['xgb'] = best_xgb
        self.best_params['xgb'] = random_search.best_params_
        
        training_time = time.time() - start_time
        self.training_times['xgb'] = training_time
        
        print(f"XGBoost Best Params: {random_search.best_params_}")
        print(f"XGBoost Best Score: {random_search.best_score_:.4f}")
        print(f"XGBoost Training Time: {training_time:.1f}s")
        
        return best_xgb
    
    def train_svm_simplified(self, X_train, y_train, cv_folds=3):
        print("\nTraining SVM Classifier (Simplified Search)...")
        
        import time
        start_time = time.time()
        
        param_grid = {
            'C': [0.1, 1, 10],
            'gamma': ['scale', 0.01],
            'kernel': ['rbf']
        }
        
        svm_model = SVC(
            probability=True,
            random_state=self.random_state,
            class_weight='balanced'
        )
        
        random_search = RandomizedSearchCV(
            svm_model, param_grid, n_iter=4,
            cv=cv_folds, scoring='f1_weighted',
            n_jobs=self.n_jobs, verbose=1, random_state=self.random_state
        )
        
        random_search.fit(X_train, y_train)
        
        best_svm = random_search.best_estimator_
        self.models['svm'] = best_svm
        self.best_params['svm'] = random_search.best_params_
        
        training_time = time.time() - start_time
        self.training_times['svm'] = training_time
        
        print(f"SVM Best Params: {random_search.best_params_}")
        print(f"SVM Best Score: {random_search.best_score_:.4f}")
        print(f"SVM Training Time: {training_time:.1f}s")
        
        return best_svm
    
    def train_lightgbm_simplified(self, X_train, y_train, cv_folds=3):
        try:
            import lightgbm as lgb
            print("\nTraining LightGBM Classifier (Fast Alternative)...")
            
            import time
            start_time = time.time()
            
            param_grid = {
                'n_estimators': [50, 100],
                'max_depth': [5, 10],
                'learning_rate': [0.05, 0.1],
                'num_leaves': [31, 50]
            }
            
            lgb_model = lgb.LGBMClassifier(
                random_state=self.random_state,
                n_jobs=self.n_jobs
            )
            
            random_search = RandomizedSearchCV(
                lgb_model, param_grid, n_iter=4,
                cv=cv_folds, scoring='f1_weighted',
                n_jobs=self.n_jobs, verbose=1, random_state=self.random_state
            )
            
            random_search.fit(X_train, y_train)
            
            best_lgb = random_search.best_estimator_
            self.models['lgb'] = best_lgb
            self.best_params['lgb'] = random_search.best_params_
            
            training_time = time.time() - start_time
            self.training_times['lgb'] = training_time
            
            print(f"LightGBM Best Params: {random_search.best_params_}")
            print(f"LightGBM Best Score: {random_search.best_score_:.4f}")
            print(f"LightGBM Training Time: {training_time:.1f}s")
            
            return best_lgb
        except ImportError:
            print("LightGBM not installed, skipping...")
            return None
    
    def evaluate_model(self, model, X_test, y_test, model_name, reverse_mapping, dataset_type="Test Set"):
        print(f"\nEvaluating {model_name} model ({dataset_type})...")
        
        y_pred = model.predict(X_test)
        
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        
        cm = confusion_matrix(y_test, y_pred)
        
        self.model_performance[model_name] = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'confusion_matrix': cm,
            'y_pred': y_pred,
            'y_test': y_test
        }
        
        print(f"{dataset_type} - Accuracy: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1 Score: {f1:.4f}")
        
        return accuracy, precision, recall, f1
    
    def train_all_models_fast(self, X_train, X_test, y_train, y_test, reverse_mapping, 
                             use_models=None):
        print("Starting fast training of supervised learning models (Transfer Learning, No Validation Set)...")
        print("="*60)
        
        if use_models is None:
            use_models = ['knn', 'rf', 'xgb', 'svm']
        
        if 'knn' in use_models:
            self.train_knn_simplified(X_train, y_train)
        
        if 'rf' in use_models:
            self.train_random_forest_simplified(X_train, y_train)
        
        if 'xgb' in use_models:
            self.train_xgboost_simplified(X_train, y_train)
        
        if 'svm' in use_models:
            self.train_svm_simplified(X_train, y_train)
        
        if 'lgb' in use_models:
            self.train_lightgbm_simplified(X_train, y_train)
        
        print("\n" + "="*60)
        print("Evaluating all models (Transfer Learning, No Validation Set)...")
        print("="*60)
        
        for model_name, model in self.models.items():
            self.evaluate_model(model, X_test, y_test, model_name, reverse_mapping, dataset_type="Test Set")
        
        self.print_training_summary()
        
        return self.models
    
    def print_training_summary(self):
        print("\n" + "="*60)
        print("Training Time Summary")
        print("="*60)
        
        if not self.training_times:
            print("No training time data")
            return
        
        total_time = sum(self.training_times.values())
        
        for model_name, time_taken in self.training_times.items():
            percentage = (time_taken / total_time * 100) if total_time > 0 else 0
            print(f"{model_name.upper():10s}: {time_taken:6.1f}s ({percentage:5.1f}%)")
        
        print(f"{'Total':10s}: {total_time:6.1f}s")
    
    def compare_models(self):
        print("\n" + "="*60)
        print("Model Performance Comparison (Transfer Learning, No Validation Set)")
        print("="*60)
        
        comparison_data = []
        
        for model_name, performance in self.model_performance.items():
            training_time = self.training_times.get(model_name, 0)
            
            row = {
                'Model': model_name.upper(),
                'Test Accuracy': f"{performance['accuracy']:.4f}",
                'Test F1 Score': f"{performance['f1']:.4f}",
                'Training Time(s)': f"{training_time:.1f}"
            }
            comparison_data.append(row)
        
        comparison_df = pd.DataFrame(comparison_data)
        print(comparison_df.to_string(index=False))
        
        return comparison_df
    
    def plot_model_comparison_simple(self):
        if not self.model_performance:
            print("No model performance data for visualization")
            return
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        model_names = [name.upper() for name in self.model_performance.keys()]
        f1_scores = [perf['f1'] for perf in self.model_performance.values()]
        
        bars1 = axes[0].bar(model_names, f1_scores, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'][:len(model_names)])
        axes[0].set_title('Model Test F1 Score Comparison (TL, No Val Set)', fontsize=14, fontweight='bold')
        axes[0].set_ylabel('F1 Score')
        axes[0].set_ylim(0, 1.0)
        axes[0].axhline(y=0.5, color='r', linestyle='--', alpha=0.5)
        
        for bar in bars1:
            height = bar.get_height()
            axes[0].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                       f'{height:.3f}', ha='center', va='bottom')
        
        training_times = [self.training_times.get(name, 0) for name in self.model_performance.keys()]
        
        bars2 = axes[1].bar(model_names, training_times, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'][:len(model_names)])
        axes[1].set_title('Model Training Time Comparison (TL, No Val Set)', fontsize=14, fontweight='bold')
        axes[1].set_ylabel('Training Time(s)')
        
        for bar in bars2:
            height = bar.get_height()
            axes[1].text(bar.get_x() + bar.get_width()/2., height + 0.5,
                       f'{height:.1f}s', ha='center', va='bottom')
        
        plt.suptitle('TL Supervised Model Performance & Efficiency Comparison (No Val Set)', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'{CONFIG["result_prefix"]}model_comparison_fast.png', dpi=300, bbox_inches='tight')
        plt.show()
    
    def get_best_model(self):
        if not self.model_performance:
            print("No model performance data")
            return None, 0, None
        
        best_model_name = None
        best_f1 = 0
        best_performance = None
        
        for model_name, performance in self.model_performance.items():
            if performance['f1'] > best_f1:
                best_f1 = performance['f1']
                best_model_name = model_name
                best_performance = performance
        
        return best_model_name, best_f1, best_performance
    
    def save_results_fast(self, reverse_mapping, sofm_data):
        print("\nSaving transfer learning results (No Validation Set)...")
        
        comparison_data = []
        for model_name, performance in self.model_performance.items():
            training_time = self.training_times.get(model_name, 0)
            
            row = {
                'Model': model_name.upper(),
                'Test Accuracy': performance['accuracy'],
                'Test Precision': performance['precision'],
                'Test Recall': performance['recall'],
                'Test F1 Score': performance['f1'],
                'Training Time(s)': training_time
            }
            comparison_data.append(row)
        
        comparison_df = pd.DataFrame(comparison_data)
        comparison_df.to_csv(f'{CONFIG["result_prefix"]}model_comparison_fast.csv', index=False, encoding='utf-8-sig')
        print(f"✓ Model comparison results saved to: {CONFIG['result_prefix']}model_comparison_fast.csv")
        
        params_data = []
        for model_name, params in self.best_params.items():
            for param_name, param_value in params.items():
                params_data.append({
                    'Model': model_name.upper(),
                    'Parameter': param_name,
                    'Value': str(param_value)
                })
        
        if params_data:
            params_df = pd.DataFrame(params_data)
            params_df.to_csv(f'{CONFIG["result_prefix"]}best_params_fast.csv', index=False, encoding='utf-8-sig')
            print(f"✓ Best parameters saved to: {CONFIG['result_prefix']}best_params_fast.csv")
        
        best_model_name, best_f1, _ = self.get_best_model()
        if best_model_name and best_model_name in self.models:
            best_model = self.models[best_model_name]
            joblib.dump(best_model, f'{CONFIG["result_prefix"]}best_model_{best_model_name}.pkl')
            print(f"✓ Best model({best_model_name.upper()}) saved to: {CONFIG['result_prefix']}best_model_{best_model_name}.pkl")
        
        classifier_data = {
            'models': self.models,
            'scaler': self.scaler,
            'best_params': self.best_params,
            'model_performance': self.model_performance,
            'training_times': self.training_times,
            'random_state': self.random_state
        }
        
        joblib.dump(classifier_data, f'{CONFIG["result_prefix"]}supervised_classifier_fast.pkl')
        print(f"✓ Complete classifier object saved to: {CONFIG['result_prefix']}supervised_classifier_fast.pkl")
    
    def quick_predict(self, features, model_name=None):
        if not self.models:
            print("Error: No trained models available")
            return None
        
        if model_name is None:
            best_model_name, _, _ = self.get_best_model()
            if best_model_name is None:
                print("Error: Cannot determine best model")
                return None
            model_name = best_model_name
        
        if model_name not in self.models:
            print(f"Error: Model '{model_name}' not found")
            print(f"Available models: {list(self.models.keys())}")
            return None
        
        features_scaled = self.scaler.transform(features.reshape(1, -1))
        
        model = self.models[model_name]
        prediction = model.predict(features_scaled)[0]
        
        return prediction

def main_fast_classification(use_models=None, cv_folds=3):
    print("Fast Supervised Learning Maturity Classification System (Transfer Learning, No Validation Set)")
    print("="*60)
    print("Training models with simplified parameter search (No Validation Set)")
    print("="*60)
    
    classifier = SupervisedMaturityClassifier(random_state=42)
    
    X_train, X_test, y_train, y_test, label_mapping, reverse_mapping, sofm_data = classifier.load_sofm_data()
    
    X_train_scaled, X_test_scaled, y_train, y_test = classifier.prepare_data(
        X_train, X_test, y_train, y_test
    )
    
    classifier.train_all_models_fast(X_train_scaled, X_test_scaled, 
                                    y_train, y_test, reverse_mapping, use_models=use_models)
    
    comparison_df = classifier.compare_models()
    
    classifier.plot_model_comparison_simple()
    
    print("\n" + "="*60)
    print("Best Model Identification (Transfer Learning, No Validation Set)")
    print("="*60)
    
    best_model_name, best_f1, best_performance = classifier.get_best_model()
    
    if best_model_name:
        print(f"Best Model: {best_model_name.upper()} (Test F1 Score: {best_f1:.4f})")
        
        if best_performance and 'confusion_matrix' in best_performance:
            cm = best_performance['confusion_matrix']
            print(f"\n{best_model_name.upper()} Test Confusion Matrix:")
            print(cm)
    else:
        print("No best model found")
        best_model_name = "None"
        best_f1 = 0
    
    classifier.save_results_fast(reverse_mapping, sofm_data)
    
    print("\n" + "="*60)
    print("Fast Supervised Learning Classification Summary (Transfer Learning, No Validation Set)")
    print("="*60)
    
    total_training_time = sum(classifier.training_times.values())
    print(f"Total Training Time: {total_training_time:.1f}s")
    
    if best_model_name != "None":
        best_training_time = classifier.training_times.get(best_model_name, 0)
        print(f"Best Model({best_model_name.upper()}) Training Time: {best_training_time:.1f}s")
        print(f"Best Model({best_model_name.upper()}) Test F1 Score: {best_f1:.4f}")
        
        print(f"\nRecommended: {best_model_name.upper()} model")
        print(f"Reason: Highest F1 score on test set ({best_f1:.4f})")
    
    return classifier, best_model_name, best_f1

def quick_start():
    print("Quick Start Supervised Learning Classification (Transfer Learning, No Validation Set)...")
    print("="*60)
    
    use_models = ['rf', 'xgb']
    
    classifier, best_model, best_f1 = main_fast_classification(
        use_models=use_models,
        cv_folds=3
    )
    
    print("\n" + "="*60)
    print("Fast Training Completed (Transfer Learning, No Validation Set)!")
    print("="*60)
    print(f"Best Model: {best_model.upper() if best_model != 'None' else 'None'} (F1: {best_f1:.4f})")
    
    return classifier

if __name__ == "__main__":
    required_files = [CONFIG["tl_sofm_path"], CONFIG["tl_feat_path"]]
    
    missing_files = [f for f in required_files if not os.path.exists(f)]
    
    if missing_files:
        print("Error: Missing required transfer learning files")
        for f in missing_files:
            print(f"  - {f}")
        print("\nPlease run transfer learning part 1, 2, 3 first")
    else:
        print("="*60)
        print("Please select run mode (Transfer Learning, No Validation Set):")
        print("1. Fast Mode (Train only RF and XGBoost, fastest)")
        print("2. Standard Mode (Train all 4 models)")
        print("3. Custom Mode (Select models to train)")
        print("="*60)
        
        choice = input("Enter selection (1/2/3): ").strip()
        
        if choice == '1':
            classifier = quick_start()
        elif choice == '2':
            classifier, best_model, best_f1 = main_fast_classification()
        elif choice == '3':
            print("\nAvailable models:")
            print("  knn  - K-Nearest Neighbors")
            print("  rf   - Random Forest (Recommended)")
            print("  xgb  - XGBoost (Recommended)")
            print("  svm  - Support Vector Machine")
            print("  lgb  - LightGBM (If installed)")
            
            model_input = input("Enter models to train (comma-separated, e.g.: rf,xgb): ").strip()
            use_models = [m.strip() for m in model_input.split(',') if m.strip()]
            
            if not use_models:
                print("No models selected, using default settings...")
                use_models = ['rf', 'xgb']
            
            classifier, best_model, best_f1 = main_fast_classification(use_models=use_models)
        else:
            print("Invalid selection, using fast mode...")
            classifier = quick_start()

Please select run mode (Transfer Learning, No Validation Set):
1. Fast Mode (Train only RF and XGBoost, fastest)
2. Standard Mode (Train all 4 models)
3. Custom Mode (Select models to train)


Enter selection (1/2/3):  2


Fast Supervised Learning Maturity Classification System (Transfer Learning, No Validation Set)
Training models with simplified parameter search (No Validation Set)
Loading transfer learning SOFM clustering results...
Labeled training samples: 945/1704
Labeled test samples: 311/568
Original labels: [np.int64(-3), np.int64(-2), np.int64(-1), np.int64(1), np.int64(2)] -> Encoded labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Class sample statistics:
Level -3(Encoded 0): Train=221, Test=70
Level -2(Encoded 1): Train=116, Test=48
Level -1(Encoded 2): Train=216, Test=74
Level 1(Encoded 3): Train=214, Test=58
Level 2(Encoded 4): Train=178, Test=61

Preparing data (Transfer Learning Version, No Validation Set)...
Training set: (945, 1024), Test set: (311, 1024)
Training set class distribution:
  Class 0: 221 samples (23.4%)
  Class 1: 116 samples (12.3%)
  Class 2: 216 samples (22.9%)
  Class 3: 214 samples (22.6%)
  Class 4: 178 samples (18.8%)
Starting fast traini